In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, make_scorer, f1_score
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# load final model data
df = pd.read_csv("model_data_final_v2.csv")

# multiclass mapping of target varible: InjurySeverity
df['InjurySeverity'].value_counts()
df['InjurySeverity'] = df['InjurySeverity'].map({5:0, 4:0, 3:1, 2:1, 1:2})
df['InjurySeverity'].value_counts()

# drop redundant cols
drop_cols = ['IncidentID', 
             'IncidentDateTime',
             'has_child',
             'first_harmful_fixed_object',
             'is_age_middle']
df.drop(drop_cols, axis = 1, inplace = True)

In [3]:
# features and target variable
X = df.drop('InjurySeverity', axis = 1).copy()
y = df['InjurySeverity'].copy()

# train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y, test_size = 0.3, random_state = 42)

In [4]:
# compute class weights and map to sample weights
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight = 'balanced', classes = classes, y = y_train)
weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
sample_weights = y_train.map(weight_dict)

In [5]:
# define model
rf = RandomForestClassifier(
    random_state = 42, 
    class_weight = None, 
    n_jobs = -1
)

# hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# stratified 5-fold CV
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

# use macro F1 as scoring metric
scorer = make_scorer(f1_score, average = 'macro')

# grid search with sample weights
grid_search = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    cv = cv,
    scoring = scorer,
    verbose = 1,
    n_jobs = -1
)

grid_search.fit(X_train, y_train, sample_weight = sample_weights)
best_model = grid_search.best_estimator_
print(f"\nBest parameters: {grid_search.best_params_}\n")

# predict on test set
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best parameters: {'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}



In [6]:
# model evaluation
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names = ['KA', 'BC', 'O'])

# confusion matrix
class_names = ['KA', 'BC', 'O']
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index = [f'Actual_{cls}' for cls in class_names],
    columns = [f'Predicted_{cls}' for cls in class_names]
)

print(f"Random Forest")
print(f"Accuracy: {round(accuracy, 4)}\n")
print(f"Classification Report: \n{report}\n")
print(f"Confusion Matrix: \n{cm_df}\n")

report_dict = classification_report(y_test, y_pred, target_names = ['KA', 'BC', 'O'], output_dict = True)

f1_scores = [
    report_dict['KA']['f1-score'],
    report_dict['BC']['f1-score'],
    report_dict['O']['f1-score']
]

harmonic_mean_f1 = len(f1_scores) / sum(1.0 / f for f in f1_scores if f > 0)
print(f"Harmonic Mean of F1-scores: {round(harmonic_mean_f1, 4)}")

Random Forest
Accuracy: 0.6117

Classification Report: 
              precision    recall  f1-score   support

          KA       0.18      0.32      0.23        34
          BC       0.40      0.36      0.38       148
           O       0.77      0.74      0.75       364

    accuracy                           0.61       546
   macro avg       0.45      0.48      0.46       546
weighted avg       0.63      0.61      0.62       546


Confusion Matrix: 
           Predicted_KA  Predicted_BC  Predicted_O
Actual_KA            11            15            8
Actual_BC            22            54           72
Actual_O             29            66          269

Harmonic Mean of F1-scores: 0.361
